In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from cvd_env.reactor_env import CVDReactorEnv
from agents.manual_policies import always_max_policy, proportional_policy, random_policy
from agents.dqn_trainer import train_dqn
from agents.compare_policies import evaluate_policy
from utils.logger import setup_logging

setup_logging()

In [ ]:
env = CVDReactorEnv(
    target_thickness=100.0,
    max_steps=20,
    k0=1e14,
    Ea=250000,
    alpha=1.0,
    dt=0.01,
    mode="real",
)

In [ ]:
model = train_dqn(env, total_timesteps=5000)


def rl_policy(env, obs):
    action, _ = model.predict(obs)
    return int(action)

In [ ]:
n_episodes = 10

errors_max = evaluate_policy(always_max_policy, env, n_episodes)
errors_prop = evaluate_policy(proportional_policy, env, n_episodes)
errors_rand = evaluate_policy(lambda env, _: random_policy(env), env, n_episodes)
errors_rl = evaluate_policy(rl_policy, env, n_episodes)

In [ ]:
import pandas as pd

results = {
    "Policy": ["Always Max", "Proportional", "Random", "RL (DQN)"],
    "Mean Error": [
        np.mean(errors_max),
        np.mean(errors_prop),
        np.mean(errors_rand),
        np.mean(errors_rl),
    ],
    "Std Error": [
        np.std(errors_max),
        np.std(errors_prop),
        np.std(errors_rand),
        np.std(errors_rl),
    ],
}

df = pd.DataFrame(results)
df